# RetinoCheck AI — Colab
Research only. Never a medical device or medical advice. Select a T4 GPU runtime. Replace REPO_URL in the setup cell with your GitHub URL. Code and data stay on /content; checkpoints persist on Drive.


In [2]:
# Paste this complete file into the first code cell of a fresh Colab session.
# Select Runtime > Change runtime type > T4 GPU before running.
from google.colab import drive
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/ratuhin1122/retinocheck-ai.git"
REPO_DIR = Path("/content/retinocheck-ai")
DRIVE_ROOT = Path("/content/drive/MyDrive/RetinoCheckAI")

if "YOUR_USERNAME" in REPO_URL or "YOUR_REPO" in REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repository URL first.")
if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("These pins target Python 3.11–3.13. Select a matching Colab runtime.")

drive.mount("/content/drive")
for folder in ("checkpoints", "reports"):
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)

# Reuse the clone when rerunning; avoid overwriting local changes with a pull.
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / ".git").exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git clone. Choose another REPO_DIR.")
else:
    remote = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
    ).strip()
    if remote != REPO_URL:
        raise RuntimeError(f"Existing clone has a different origin: {remote}")
    print("Using existing clone. Commit:", subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# CUDA 12.4 wheels: the official matching PyTorch 2.6 / torchvision 0.21 pair.
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch==2.6.0", "torchvision==0.21.0",
                "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

# A separate Python process checks the newly installed packages, even if the
# notebook kernel already imported older versions. Allocate a GPU tensor too.
subprocess.run([sys.executable, "-c", """
import torch, torchvision, timm, datasets, sklearn, gradio, pytorch_grad_cam
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU. Select a T4 GPU runtime and rerun setup.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU tensor check:', (torch.ones(1, device='cuda') + 1).item())
"""], check=True)
print("Setup complete. If packages were already imported, restart the session before training.")


Mounted at /content/drive
Setup complete. If packages were already imported, restart the session before training.


If pip upgraded an already-imported package, restart the session and rerun setup. The download/cache stays on Colab local disk; checkpoints go to mounted Drive.

In [3]:
import torch
from src.config import Config, CHECKPOINT_ROOT
from src.data import load_source
from src.train import train
from src.evaluate import evaluate
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — select a T4 runtime")
MODEL = "resnet18"  # Or "efficientnet_b0"; each model gets its own folder.
config = Config(model_name=MODEL, output_dir=str(CHECKPOINT_ROOT / MODEL))

GPU: Tesla T4


Verify a real Hugging Face download, schema, labels and one decoded image. Failure raises an actionable error. Fallback: download the original Kaggle APTOS 2019 data, extract `train.csv` and `train_images/`, then set `config.local_data = "/content/aptos2019"` and choose a distinct output directory.

In [4]:
source, labels, identity = load_source(config)
print("Loaded images:", len(labels), "fingerprint:", identity)
print("Grade counts:", {grade: int((labels == grade).sum()) for grade in range(5)})
del source

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/250M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3662 [00:00<?, ? examples/s]

Loaded images: 3662 fingerprint: 9feabdc595016e26
Grade counts: {0: 1805, 1: 370, 2: 999, 3: 193, 4: 295}


Train or resume. After disconnecting, rerun setup and configuration, set `RESUME = True`, and rerun this cell. A partial epoch restarts from the previous completed epoch. Do not reuse a run folder for a different experiment.

In [5]:
RESUME = False
history = train(config, resume=f"{config.output_dir}/last.pt" if RESUME else None)

Saved frozen splits: /content/drive/MyDrive/RetinoCheckAI/splits/aptos_indices.json


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Epoch 1/15:   0%|          | 0/81 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 1.5214683538389262, 'val_loss': 1.4318016179055246, 'l1_lambda': 0.0, 'qwk': 0.6879409128792766, 'sample_count': 549, 'class_counts': [271, 55, 150, 29, 44], 'accuracy': 0.6830601092896175, 'macro_f1': 0.4753550560559727, 'macro_recall': 0.47870949731440876, 'per_class_recall': [0.8782287822878229, 0.4909090909090909, 0.6466666666666666, 0.2413793103448276, 0.13636363636363635], 'confusion_matrix': [[238, 14, 15, 4, 0], [6, 27, 21, 1, 0], [14, 30, 97, 5, 4], [1, 1, 18, 7, 2], [2, 10, 20, 6, 6]]}


Epoch 2/15:   0%|          | 0/81 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b38041a1c60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1618, in __del__
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1601, in _shutdown_workers
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b38041a1c60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1618, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1601, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a chi

... (30 outputs left out)

{'epoch': 14, 'train_loss': 0.7814484134438537, 'val_loss': 0.9193912876976861, 'l1_lambda': 0.0, 'qwk': 0.8280634959624182, 'sample_count': 549, 'class_counts': [271, 55, 150, 29, 44], 'accuracy': 0.75591985428051, 'macro_f1': 0.6074056855109167, 'macro_recall': 0.630814854230047, 'per_class_recall': [0.9704797047970479, 0.5636363636363636, 0.5666666666666667, 0.6896551724137931, 0.36363636363636365], 'confusion_matrix': [[263, 5, 2, 0, 1], [13, 31, 10, 1, 0], [11, 19, 85, 30, 5], [1, 0, 7, 20, 1], [0, 7, 12, 9, 16]]}


Epoch 15/15:   0%|          | 0/81 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b38041a1c60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1618, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1601, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b38041a1c60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1618, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1601, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

{'epoch': 15, 'train_loss': 0.767935362519857, 'val_loss': 0.9045549668575246, 'l1_lambda': 0.0, 'qwk': 0.8286083982870698, 'sample_count': 549, 'class_counts': [271, 55, 150, 29, 44], 'accuracy': 0.7540983606557377, 'macro_f1': 0.6102591808325861, 'macro_recall': 0.6381765896655832, 'per_class_recall': [0.959409594095941, 0.6181818181818182, 0.56, 0.6896551724137931, 0.36363636363636365], 'confusion_matrix': [[260, 8, 2, 0, 1], [10, 34, 10, 1, 0], [11, 18, 84, 33, 4], [1, 0, 7, 20, 1], [0, 7, 12, 9, 16]]}
Early stopping: QWK did not improve for 5 consecutive epochs.


Save package versions alongside the run. Compare validation histories between models and choose settings before accessing test results.

In [6]:
import subprocess
from pathlib import Path
Path(config.output_dir, "environment.txt").write_text(subprocess.check_output(["pip", "freeze"], text=True))

14275

Final held-out evaluation: run once for the model selected using validation results. Report all metrics, class counts and splitting limitations.

In [7]:
results = evaluate(f"{config.output_dir}/best.pt", output_dir=f"{config.output_dir}/reports")
print(results)

Reusing frozen splits: /content/drive/MyDrive/RetinoCheckAI/splits/aptos_indices.json
{'qwk': 0.8569701203273268, 'sample_count': 550, 'class_counts': [271, 56, 150, 29, 44], 'accuracy': 0.7490909090909091, 'macro_f1': 0.5967678880855108, 'macro_recall': 0.6333116183557715, 'per_class_recall': [0.977859778597786, 0.6428571428571429, 0.49333333333333335, 0.6206896551724138, 0.4318181818181818], 'confusion_matrix': [[265, 6, 0, 0, 0], [5, 36, 12, 2, 1], [7, 20, 74, 42, 7], [1, 0, 1, 18, 9], [2, 3, 6, 14, 19]]}


Optional research demo. Running this cell creates a temporary public Gradio link because Colab cannot expose a localhost UI directly. Use public or de-identified research images. Close the demo/runtime after use.

In [8]:
from app.gradio_app import create_app
demo = create_app(f"{config.output_dir}/best.pt")
demo.launch(share=True)

Loading checkpoint from: /content/drive/MyDrive/RetinoCheckAI/checkpoints/resnet18/best.pt
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0f9f91115c179ef033.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
